# Cluster Analysis

Differences between groups were tested using a Chi-squared test for categorical variables and a one-way ANOVA for continuous variables. 

In [1]:
%reload_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import seaborn as sn

from IPython.display import display

from preproc import get_hfpef_405, convert2np

from scipy.stats import f_oneway, chi2_contingency

from sklearn.metrics.cluster import contingency_matrix

np.set_printoptions(precision=3, suppress=True)
pd.options.display.float_format = "{:,.3f}".format

In [2]:
data_df, patient_df = get_hfpef_405(get_patient_no=True)

In [4]:
data_df.describe()

,Age,Sex,Cr,GFR,CKD stage,smoke,BMI,BSA,DM,Insulin,...,E/A,Mitral E/e',TR Vmax,RWT,LV mass index,LVH,LAVI,LACI,LA diameter,LVEF
count,398.000,398.000,398.000,398.000,398.000,398.000,398.000,398.000,398.000,398.000,...,398.000,398.000,398.000,398.000,398.000,398.000,398.000,398.000,398.000,398.000
mean,72.369,1.673,1.561,54.485,2.724,0.070,26.756,1.636,0.583,0.186,...,0.725,16.784,1.850,0.586,113.988,2.724,35.116,3.473,3.834,62.252
std,11.212,0.470,1.318,24.351,0.996,0.256,6.617,0.231,0.494,0.390,...,0.589,8.684,1.415,0.348,51.265,1.991,26.129,4.026,1.319,8.656
min,31.000,1.000,0.550,2.000,1.000,0.000,15.400,1.120,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
25%,65.000,1.000,0.880,38.000,2.000,0.000,22.400,1.500,0.000,0.000,...,0.480,11.200,0.000,0.370,86.900,1.000,18.375,0.000,3.600,57.000
50%,74.000,2.000,1.160,55.500,3.000,0.000,26.000,1.600,1.000,0.000,...,0.700,15.850,2.440,0.500,113.000,3.000,35.300,2.734,4.000,62.000
75%,81.000,2.000,1.580,71.000,3.000,0.000,30.375,1.800,1.000,0.000,...,1.000,22.100,2.998,0.720,141.635,4.000,50.500,5.606,4.482,67.000
max,97.000,2.000,10.000,108.000,5.000,1.000,55.900,2.300,1.000,1.000,...,4.100,42.200,4.550,2.300,356.600,7.000,140.800,23.665,7.000,82.900


## Get Cluster

In [5]:
from methods import get_sc_pred, get_p_ci
from utils import get_score, score_columns, plot_clustering_score, plot_contingency_matrix, plot_bic_aic

import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('ggplot')

In [6]:
lbl_colname= ['Death', 'CV death', 'Major cardiac events', 'HF re-hospitalization']

## Analysis

In [7]:
X, y, feature_list = convert2np(data_df, lbl_colname)
y_pred = get_sc_pred(3, X)

feat_df, cluster_df = get_p_ci(data_df, y_pred)
display(feat_df)
display(cluster_df)
# cluster_df.to_excel("HFpEF_allfeat_spectral_3clus_405samples.xlsx")  

C:\Users\sguly\Documents\HFpEF\preproc.py:55: UserWarning: Warning...........selected_feat is not given, set to 0 where all features are used.
  warnings.warn("Warning...........selected_feat is not given, set to 0 where all features are used.")


,p-value,is_category
Age,0.211,False
Sex,0.643,True
Cr,0.000,False
GFR,0.000,False
CKD stage,0.000,True
smoke,0.341,True
BMI,0.000,False
BSA,0.037,False
DM,0.567,True
Insulin,0.806,True


,0_mu,0_ci_l,0_ci_h,1_mu,1_ci_l,1_ci_h,2_mu,2_ci_l,2_ci_h,p-value
n,146.000,0.000,0.000,171.000,0.000,0.000,81.000,0.000,0.000,0.000
Sex_1,46.000,31.507,0.000,54.000,31.579,0.000,30.000,37.037,0.000,0.643
Sex_2,100.000,68.493,0.000,117.000,68.421,0.000,51.000,62.963,0.000,0.643
CKD stage_1,17.000,11.644,0.000,17.000,9.942,0.000,0.000,0.000,0.000,0.000
CKD stage_2,48.000,32.877,0.000,65.000,38.012,0.000,24.000,29.630,0.000,0.000
...,...,...,...,...,...,...,...,...,...,...
LV mass index,109.508,101.595,117.422,110.395,102.309,118.481,129.647,119.362,139.932,0.008
LAVI,37.238,32.666,41.810,33.687,30.130,37.243,34.307,28.389,40.226,0.461
LACI,3.407,2.711,4.102,3.153,2.665,3.641,4.267,3.205,5.328,0.118
LA diameter,4.005,3.762,4.248,3.769,3.595,3.944,3.663,3.389,3.938,0.120


In [8]:
patient_df['cluster'] = y_pred

In [9]:
with pd.ExcelWriter("HFpEF_allfeat_spectral_3clus_405samples.xlsx", engine='xlsxwriter') as writer:    
    cluster_df.to_excel(writer, 'analysis')   
    patient_df.to_excel(writer, 'cluster')

C:\Users\sguly\miniconda3\envs\hfpef\lib\site-packages\xlsxwriter\workbook.py:339: UserWarning: Calling close() on already closed file.
  warn("Calling close() on already closed file.")


In [10]:
# X, y, feature_list = convert2np(data_df, lbl_colname, selected_feat=True)
# y_pred = get_sc_pred(2, X)

# feat_df, cluster_df = get_p_ci(data_df, y_pred)
# display(feat_df)
# display(cluster_df)
# cluster_df.to_excel("HFpEF_selfeat_spectral_2clus_405samples.xlsx")  

In [11]:
X, y, feature_list = convert2np(data_df, lbl_colname, selected_feat=1)
y_pred = get_sc_pred(3, X)

feat_df, cluster_df = get_p_ci(data_df, y_pred)
display(feat_df)
display(cluster_df)
# cluster_df.to_excel("HFpEF_selfeat_spectral_3clus_405samples.xlsx")  

,p-value,is_category
Age,0.113,False
Sex,0.272,True
Cr,0.000,False
GFR,0.000,False
CKD stage,0.000,True
smoke,0.201,True
BMI,0.000,False
BSA,0.141,False
DM,0.324,True
Insulin,0.744,True


,0_mu,0_ci_l,0_ci_h,1_mu,1_ci_l,1_ci_h,2_mu,2_ci_l,2_ci_h,p-value
n,137.000,0.000,0.000,170.000,0.000,0.000,91.000,0.000,0.000,0.000
Sex_1,41.000,29.927,0.000,53.000,31.176,0.000,36.000,39.560,0.000,0.272
Sex_2,96.000,70.073,0.000,117.000,68.824,0.000,55.000,60.440,0.000,0.272
CKD stage_1,16.000,11.679,0.000,17.000,10.000,0.000,1.000,1.099,0.000,0.000
CKD stage_2,45.000,32.847,0.000,65.000,38.235,0.000,27.000,29.670,0.000,0.000
...,...,...,...,...,...,...,...,...,...,...
LV mass index,106.387,98.257,114.517,110.643,102.524,118.762,131.680,122.203,141.157,0.001
LAVI,36.336,31.574,41.099,33.730,30.154,37.306,35.866,30.353,41.379,0.654
LACI,3.097,2.459,3.735,3.152,2.661,3.642,4.638,3.555,5.721,0.007
LA diameter,3.957,3.703,4.212,3.770,3.595,3.946,3.769,3.510,4.028,0.405


In [12]:
patient_df['cluster'] = y_pred

In [13]:
with pd.ExcelWriter("HFpEF_selfeat_spectral_3clus_405samples.xlsx", engine='xlsxwriter') as writer:    
    cluster_df.to_excel(writer, 'analysis')   
    patient_df.to_excel(writer, 'cluster')

C:\Users\sguly\miniconda3\envs\hfpef\lib\site-packages\xlsxwriter\workbook.py:339: UserWarning: Calling close() on already closed file.
  warn("Calling close() on already closed file.")


In [14]:
X, y, feature_list = convert2np(data_df, lbl_colname, selected_feat=2)
y_pred = get_sc_pred(3, X)

feat_df, cluster_df = get_p_ci(data_df, y_pred)
display(feat_df)
display(cluster_df)
# cluster_df.to_excel("HFpEF_onefeat_spectral_3clus_405samples.xlsx")  

,p-value,is_category
Age,0.222,False
Sex,0.422,True
Cr,0.000,False
GFR,0.000,False
CKD stage,0.000,True
smoke,0.273,True
BMI,0.000,False
BSA,0.130,False
DM,0.429,True
Insulin,0.603,True


,0_mu,0_ci_l,0_ci_h,1_mu,1_ci_l,1_ci_h,2_mu,2_ci_l,2_ci_h,p-value
n,141.000,0.000,0.000,154.000,0.000,0.000,103.000,0.000,0.000,0.000
Sex_1,44.000,31.206,0.000,47.000,30.519,0.000,39.000,37.864,0.000,0.422
Sex_2,97.000,68.794,0.000,107.000,69.481,0.000,64.000,62.136,0.000,0.422
CKD stage_1,19.000,13.475,0.000,13.000,8.442,0.000,2.000,1.942,0.000,0.000
CKD stage_2,49.000,34.752,0.000,58.000,37.662,0.000,30.000,29.126,0.000,0.000
...,...,...,...,...,...,...,...,...,...,...
LV mass index,104.483,96.585,112.381,110.043,101.334,118.752,132.897,124.227,141.568,0.000
LAVI,35.640,30.928,40.353,34.115,30.371,37.859,35.893,30.809,40.977,0.830
LACI,3.152,2.513,3.791,3.092,2.585,3.600,4.481,3.503,5.459,0.012
LA diameter,3.890,3.634,4.145,3.785,3.602,3.969,3.832,3.599,4.064,0.795


In [15]:
patient_df['cluster'] = y_pred

In [16]:
with pd.ExcelWriter("HFpEF_onefeat_spectral_3clus_405samples.xlsx", engine='xlsxwriter') as writer:    
    cluster_df.to_excel(writer, 'analysis')   
    patient_df.to_excel(writer, 'cluster')

C:\Users\sguly\miniconda3\envs\hfpef\lib\site-packages\xlsxwriter\workbook.py:339: UserWarning: Calling close() on already closed file.
  warn("Calling close() on already closed file.")
